In [1]:
# W7D1 — Haystack Pipeline Architecture
# Cell 1: Install required packages

!pip install -q haystack-ai pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.7/665.7 kB 23.5 MB/s eta 0:00:00


In [2]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

print("✅ Haystack imported successfully!")
print("Haystack pipeline components are ready.")

✅ Haystack imported successfully!
Haystack pipeline components are ready.


In [3]:
# Create an in-memory document store

document_store = InMemoryDocumentStore()

print("✅ Document Store created successfully!")
print("Document count:", document_store.count_documents())

✅ Document Store created successfully!
Document count: 0


In [4]:
documents = [
    Document(
        content="Python is a high-level programming language used for web development, automation, data science, artificial intelligence, and machine learning.",
        meta={"topic": "Python", "source": "document_1"}
    ),
    Document(
        content="Machine learning is a branch of artificial intelligence where computers learn patterns from data and use those patterns to make predictions or decisions.",
        meta={"topic": "Machine Learning", "source": "document_2"}
    ),
    Document(
        content="A database is an organized collection of data. SQL databases use tables, rows, and columns to store and manage structured information.",
        meta={"topic": "Database", "source": "document_3"}
    ),
    Document(
        content="Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. Relevant documents are retrieved first and then provided to the language model as context.",
        meta={"topic": "RAG", "source": "document_4"}
    ),
    Document(
        content="Haystack is an open-source framework for building search and question-answering applications. It provides components for document stores, retrievers, generators, and pipelines.",
        meta={"topic": "Haystack", "source": "document_5"}
    )
]

print("✅ 5 documents created successfully!")

for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc.meta['topic']} → {doc.content[:80]}...")

✅ 5 documents created successfully!
1. Python → Python is a high-level programming language used for web development, automation...
2. Machine Learning → Machine learning is a branch of artificial intelligence where computers learn pa...
3. Database → A database is an organized collection of data. SQL databases use tables, rows, a...
4. RAG → Retrieval-Augmented Generation, or RAG, combines information retrieval with a la...
5. Haystack → Haystack is an open-source framework for building search and question-answering ...


In [5]:
document_store.write_documents(documents)

print("✅ Documents indexed successfully!")
print("Total documents:", document_store.count_documents())

✅ Documents indexed successfully!
Total documents: 5


In [6]:
# Create BM25 retriever

bm25_retriever = InMemoryBM25Retriever(document_store=document_store)

print("✅ BM25 Retriever created successfully!")

✅ BM25 Retriever created successfully!


In [7]:
# Create Haystack BM25 pipeline

bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    "bm25_retriever",
    bm25_retriever
)

print("✅ BM25 Haystack pipeline created!")
print("Pipeline component: BM25 Retriever")

✅ BM25 Haystack pipeline created!
Pipeline component: BM25 Retriever


In [8]:
query = "What is Python used for?"

result = bm25_pipeline.run({
    "bm25_retriever": {
        "query": query,
        "top_k": 3
    }
})

documents_found = result["bm25_retriever"]["documents"]

print("=" * 70)
print("BM25 RETRIEVAL TEST")
print("=" * 70)
print("Question:", query)
print()

for i, doc in enumerate(documents_found, 1):
    print(f"Result {i}:")
    print("Topic:", doc.meta["topic"])
    print("Score:", doc.score)
    print("Content:", doc.content)
    print("-" * 70)

BM25 RETRIEVAL TEST
Question: What is Python used for?

Result 1:
Topic: Python
Score: 5.12169003113404
Content: Python is a high-level programming language used for web development, automation, data science, artificial intelligence, and machine learning.
----------------------------------------------------------------------
Result 2:
Topic: Haystack
Score: 3.449852041506471
Content: Haystack is an open-source framework for building search and question-answering applications. It provides components for document stores, retrievers, generators, and pipelines.
----------------------------------------------------------------------
Result 3:
Topic: Database
Score: 2.6454385445940787
Content: A database is an organized collection of data. SQL databases use tables, rows, and columns to store and manage structured information.
----------------------------------------------------------------------


In [9]:
bm25_questions = [
    "What is Python used for?",
    "What is machine learning?",
    "How does a database store data?",
    "What is RAG?",
    "What is Haystack?",
    "Which technology is used for artificial intelligence?",
    "What does a language model use as context in RAG?",
    "What is used to manage structured data?",
    "Which framework can be used to build search applications?",
    "How can computers learn patterns from data?"
]

bm25_results = []

print("=" * 80)
print("BM25 RETRIEVAL — 10 QUESTIONS")
print("=" * 80)

for i, question in enumerate(bm25_questions, 1):
    result = bm25_pipeline.run({
        "bm25_retriever": {
            "query": question,
            "top_k": 1
        }
    })

    docs = result["bm25_retriever"]["documents"]

    if docs:
        top_doc = docs[0]

        bm25_results.append({
            "question": question,
            "retrieved_topic": top_doc.meta["topic"],
            "score": top_doc.score
        })

        print(f"\nQ{i}. {question}")
        print("Retrieved:", top_doc.meta["topic"])
        print("Score:", top_doc.score)
    else:
        print(f"\nQ{i}. {question}")
        print("No document retrieved.")

print("\n" + "=" * 80)
print("✅ Completed all 10 BM25 questions")

BM25 RETRIEVAL — 10 QUESTIONS

Q1. What is Python used for?
Retrieved: Python
Score: 5.12169003113404

Q2. What is machine learning?
Retrieved: Python
Score: 2.652913722300058

Q3. How does a database store data?
Retrieved: Database
Score: 4.5716499921719285

Q4. What is RAG?
Retrieved: RAG
Score: 1.8482237443574379

Q5. What is Haystack?
Retrieved: Haystack
Score: 2.0786436441101976

Q6. Which technology is used for artificial intelligence?
Retrieved: Python
Score: 5.5962105080555045

Q7. What does a language model use as context in RAG?
Retrieved: RAG
Score: 9.312140924823073

Q8. What is used to manage structured data?
Retrieved: Database
Score: 6.122696458721846

Q9. Which framework can be used to build search applications?
Retrieved: Haystack
Score: 6.367558635010317

Q10. How can computers learn patterns from data?
Retrieved: Machine Learning
Score: 7.986705642195909

✅ Completed all 10 BM25 questions


In [10]:
expected_topics = [
    "Python",
    "Machine Learning",
    "Database",
    "RAG",
    "Haystack",
    "Machine Learning",
    "RAG",
    "Database",
    "Haystack",
    "Machine Learning"
]

print("=" * 80)
print("BM25 MANUAL EVALUATION")
print("=" * 80)

correct = 0

for i, result in enumerate(bm25_results):
    expected = expected_topics[i]
    retrieved = result["retrieved_topic"]

    is_correct = retrieved == expected

    if is_correct:
        correct += 1

    symbol = "✅" if is_correct else "❌"

    print(f"\nQ{i+1}: {result['question']}")
    print(f"Expected : {expected}")
    print(f"Retrieved: {retrieved}")
    print(f"Result   : {symbol}")

precision = correct / len(bm25_results)

print("\n" + "=" * 80)
print(f"BM25 Correct Results: {correct}/{len(bm25_results)}")
print(f"BM25 Precision: {precision:.2f}")
print("=" * 80)

BM25 MANUAL EVALUATION

Q1: What is Python used for?
Expected : Python
Retrieved: Python
Result   : ✅

Q2: What is machine learning?
Expected : Machine Learning
Retrieved: Python
Result   : ❌

Q3: How does a database store data?
Expected : Database
Retrieved: Database
Result   : ✅

Q4: What is RAG?
Expected : RAG
Retrieved: RAG
Result   : ✅

Q5: What is Haystack?
Expected : Haystack
Retrieved: Haystack
Result   : ✅

Q6: Which technology is used for artificial intelligence?
Expected : Machine Learning
Retrieved: Python
Result   : ❌

Q7: What does a language model use as context in RAG?
Expected : RAG
Retrieved: RAG
Result   : ✅

Q8: What is used to manage structured data?
Expected : Database
Retrieved: Database
Result   : ✅

Q9: Which framework can be used to build search applications?
Expected : Haystack
Retrieved: Haystack
Result   : ✅

Q10: How can computers learn patterns from data?
Expected : Machine Learning
Retrieved: Machine Learning
Result   : ✅

BM25 Correct Results: 8/10
BM25

In [12]:
!pip install -q sentence-transformers-haystack

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.1 MB/s eta 0:00:00


In [13]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder
)

from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

print("✅ Dense retrieval components imported successfully!")

✅ Dense retrieval components imported successfully!


In [14]:
# Create document embedder
document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

document_embedder.warm_up()

print("✅ Document embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Document embedding model loaded!


In [15]:
# Generate embeddings for all documents

embedded_documents = document_embedder.run(documents)

documents_with_embeddings = embedded_documents["documents"]

print("✅ Document embeddings generated!")
print("Number of documents:", len(documents_with_embeddings))
print(
    "Embedding dimension:",
    len(documents_with_embeddings[0].embedding)
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Document embeddings generated!
Number of documents: 5
Embedding dimension: 384


In [16]:
dense_document_store = InMemoryDocumentStore()

dense_document_store.write_documents(documents_with_embeddings)

print("✅ Dense Document Store created!")
print("Documents:", dense_document_store.count_documents())

✅ Dense Document Store created!
Documents: 5


In [17]:
# Create dense retriever

dense_retriever = InMemoryEmbeddingRetriever(
    document_store=dense_document_store
)

print("✅ Dense Retriever created successfully!")

✅ Dense Retriever created successfully!


In [18]:
# Create text embedder for user queries

text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

text_embedder.warm_up()

print("✅ Text embedding model ready!")

✅ Text embedding model ready!


In [19]:
# Create Dense Retrieval Pipeline

dense_pipeline = Pipeline()

dense_pipeline.add_component(
    "text_embedder",
    text_embedder
)

dense_pipeline.add_component(
    "dense_retriever",
    dense_retriever
)

dense_pipeline.connect(
    "text_embedder.embedding",
    "dense_retriever.query_embedding"
)

print("✅ Dense Haystack pipeline created!")
print("Pipeline: Text Embedder → Dense Retriever")

✅ Dense Haystack pipeline created!
Pipeline: Text Embedder → Dense Retriever


In [20]:
query = "What is Python used for?"

result = dense_pipeline.run({
    "text_embedder": {
        "text": query
    },
    "dense_retriever": {
        "top_k": 3
    }
})

documents_found = result["dense_retriever"]["documents"]

print("=" * 70)
print("DENSE RETRIEVAL TEST")
print("=" * 70)
print("Question:", query)
print()

for i, doc in enumerate(documents_found, 1):
    print(f"Result {i}:")
    print("Topic:", doc.meta["topic"])
    print("Score:", doc.score)
    print("Content:", doc.content)
    print("-" * 70)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

DENSE RETRIEVAL TEST
Question: What is Python used for?

Result 1:
Topic: Python
Score: 0.8198652530409329
Content: Python is a high-level programming language used for web development, automation, data science, artificial intelligence, and machine learning.
----------------------------------------------------------------------
Result 2:
Topic: Haystack
Score: 0.2571304961126361
Content: Haystack is an open-source framework for building search and question-answering applications. It provides components for document stores, retrievers, generators, and pipelines.
----------------------------------------------------------------------
Result 3:
Topic: Machine Learning
Score: 0.22788967023566484
Content: Machine learning is a branch of artificial intelligence where computers learn patterns from data and use those patterns to make predictions or decisions.
----------------------------------------------------------------------


In [21]:
dense_results = []

print("=" * 80)
print("DENSE RETRIEVAL — 10 QUESTIONS")
print("=" * 80)

for i, question in enumerate(bm25_questions, 1):

    result = dense_pipeline.run({
        "text_embedder": {
            "text": question
        },
        "dense_retriever": {
            "top_k": 1
        }
    })

    docs = result["dense_retriever"]["documents"]

    if docs:
        top_doc = docs[0]

        dense_results.append({
            "question": question,
            "retrieved_topic": top_doc.meta["topic"],
            "score": top_doc.score
        })

        print(f"\nQ{i}. {question}")
        print("Retrieved:", top_doc.meta["topic"])
        print("Score:", top_doc.score)
    else:
        print(f"\nQ{i}. {question}")
        print("No document retrieved.")

print("\n" + "=" * 80)
print("✅ Completed all 10 Dense Retrieval questions")

DENSE RETRIEVAL — 10 QUESTIONS


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q1. What is Python used for?
Retrieved: Python
Score: 0.8198652530409329


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q2. What is machine learning?
Retrieved: Machine Learning
Score: 0.84579697756309


...


Q8. What is used to manage structured data?
Retrieved: Database
Score: 0.5349115901080086


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q9. Which framework can be used to build search applications?
Retrieved: Haystack
Score: 0.5304861772805485


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q10. How can computers learn patterns from data?
Retrieved: Machine Learning
Score: 0.6456883407147509

✅ Completed all 10 Dense Retrieval questions


In [22]:
print("=" * 80)
print("DENSE RETRIEVAL MANUAL EVALUATION")
print("=" * 80)

dense_correct = 0

for i, result in enumerate(dense_results):
    expected = expected_topics[i]
    retrieved = result["retrieved_topic"]

    is_correct = retrieved == expected

    if is_correct:
        dense_correct += 1

    symbol = "✅" if is_correct else "❌"

    print(f"\nQ{i+1}: {result['question']}")
    print(f"Expected : {expected}")
    print(f"Retrieved: {retrieved}")
    print(f"Result   : {symbol}")

dense_precision = dense_correct / len(dense_results)

print("\n" + "=" * 80)
print(f"Dense Correct Results: {dense_correct}/{len(dense_results)}")
print(f"Dense Precision: {dense_precision:.2f}")
print("=" * 80)

DENSE RETRIEVAL MANUAL EVALUATION

Q1: What is Python used for?
Expected : Python
Retrieved: Python
Result   : ✅

Q2: What is machine learning?
Expected : Machine Learning
Retrieved: Machine Learning
Result   : ✅

Q3: How does a database store data?
Expected : Database
Retrieved: Database
Result   : ✅

Q4: What is RAG?
Expected : RAG
Retrieved: RAG
Result   : ✅

Q5: What is Haystack?
Expected : Haystack
Retrieved: Haystack
Result   : ✅

Q6: Which technology is used for artificial intelligence?
Expected : Machine Learning
Retrieved: Machine Learning
Result   : ✅

Q7: What does a language model use as context in RAG?
Expected : RAG
Retrieved: RAG
Result   : ✅

Q8: What is used to manage structured data?
Expected : Database
Retrieved: Database
Result   : ✅

Q9: Which framework can be used to build search applications?
Expected : Haystack
Retrieved: Haystack
Result   : ✅

Q10: How can computers learn patterns from data?
Expected : Machine Learning
Retrieved: Machine Learning
Result   : ✅



In [23]:
print("=" * 80)
print("BM25 vs DENSE RETRIEVAL COMPARISON")
print("=" * 80)

print(f"\nBM25 Precision   : {precision:.2f}")
print(f"Dense Precision  : {dense_precision:.2f}")

print("\nDetailed Comparison:")
print("-" * 80)

for i in range(len(bm25_questions)):
    bm25_topic = bm25_results[i]["retrieved_topic"]
    dense_topic = dense_results[i]["retrieved_topic"]
    expected = expected_topics[i]

    bm25_ok = bm25_topic == expected
    dense_ok = dense_topic == expected

    print(f"\nQ{i+1}: {bm25_questions[i]}")
    print(f"Expected : {expected}")
    print(f"BM25     : {bm25_topic} {'✅' if bm25_ok else '❌'}")
    print(f"Dense    : {dense_topic} {'✅' if dense_ok else '❌'}")

print("\n" + "=" * 80)
print("Comparison completed successfully! ✅")
print("=" * 80)

BM25 vs DENSE RETRIEVAL COMPARISON

BM25 Precision   : 0.80
Dense Precision  : 1.00

Detailed Comparison:
--------------------------------------------------------------------------------

Q1: What is Python used for?
Expected : Python
BM25     : Python ✅
Dense    : Python ✅

Q2: What is machine learning?
Expected : Machine Learning
BM25     : Python ❌
Dense    : Machine Learning ✅

Q3: How does a database store data?
Expected : Database
BM25     : Database ✅
Dense    : Database ✅

Q4: What is RAG?
Expected : RAG
BM25     : RAG ✅
Dense    : RAG ✅

Q5: What is Haystack?
Expected : Haystack
BM25     : Haystack ✅
Dense    : Haystack ✅

Q6: Which technology is used for artificial intelligence?
Expected : Machine Learning
BM25     : Python ❌
Dense    : Machine Learning ✅

Q7: What does a language model use as context in RAG?
Expected : RAG
BM25     : RAG ✅
Dense    : RAG ✅

Q8: What is used to manage structured data?
Expected : Database
BM25     : Database ✅
Dense    : Database ✅

Q9: Which 

In [24]:
checklist = {
    "Haystack installed": True,
    "DocumentStore created": True,
    "5 documents indexed": document_store.count_documents() == 5,
    "BM25 Retriever created": True,
    "BM25 Pipeline tested": True,
    "10 BM25 questions tested": len(bm25_results) == 10,
    "BM25 manually evaluated": True,
    "Dense Retriever created": True,
    "Dense Pipeline tested": True,
    "10 Dense questions tested": len(dense_results) == 10,
    "BM25 vs Dense compared": True
}

print("=" * 80)
print("W7D1 SELF-REVIEW CHECKLIST")
print("=" * 80)

for item, status in checklist.items():
    print(("✅" if status else "❌"), item)

print("=" * 80)

if all(checklist.values()):
    print("🎉 W7D1 PRACTICAL COMPLETED SUCCESSFULLY!")
else:
    print("⚠️ Please review the failed items.")

W7D1 SELF-REVIEW CHECKLIST
✅ Haystack installed
✅ DocumentStore created
✅ 5 documents indexed
✅ BM25 Retriever created
✅ BM25 Pipeline tested
✅ 10 BM25 questions tested
✅ BM25 manually evaluated
✅ Dense Retriever created
✅ Dense Pipeline tested
✅ 10 Dense questions tested
✅ BM25 vs Dense compared
🎉 W7D1 PRACTICAL COMPLETED SUCCESSFULLY!
